# Data Collection: Topic Modeling on Newsgroup Posts

This notebook loads the 20 Newsgroups dataset from scikit-learn, performs initial cleaning, and uploads the cleaned data to S3 as a parquet file for downstream analysis.

**Dataset**: 20 Newsgroups — ~18,800 posts from 20 different online discussion groups covering topics like sports, politics, science, and technology.  
**Output**: Cleaned parquet saved to `s3://topic-modeling-demo/00_data_collection/newsgroups.parquet`

## Imports

In [ ]:
import boto3
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

print('All imports successful')

## Constants

In [ ]:
# S3 configuration
str_bucket = 'topic-modeling-demo'
str_region = 'us-east-1'

# output filename
str_output_filename = 'newsgroups.parquet'
str_s3_output_path = f'00_data_collection/{str_output_filename}'

# AWS client
s3_client = boto3.client('s3', region_name=str_region)

print(f'S3 Bucket: {str_bucket}')
print(f'  Output Path: {str_s3_output_path}')

## Load 20 Newsgroups Dataset

In [ ]:
# load all posts, removing headers/footers/quotes so models must rely on content
data = fetch_20newsgroups(
    subset='all',
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)

print(f'Loaded {len(data.data):,} posts')
print(f'Number of categories: {len(data.target_names)}')
print(f'\nCategories:')
for int_i, str_name in enumerate(data.target_names):
    print(f'  {int_i}: {str_name}')

## Build DataFrame

In [ ]:
# create dataframe
df_raw = pd.DataFrame({
    'text': data.data,
    'category_id': data.target
})

# map category IDs to names
df_raw['category'] = df_raw['category_id'].map(lambda x: data.target_names[x])

# create broad group labels (the part before the dot)
df_raw['broad_category'] = df_raw['category'].apply(lambda x: x.split('.')[0])

print(f'DataFrame shape: {df_raw.shape}')
print(f'\nColumns: {list(df_raw.columns)}')
print(f'\nSample:')
df_raw.head()

## Data Cleaning

In [ ]:
int_initial_rows = len(df_raw)

# remove empty or whitespace-only posts
df_clean = df_raw[df_raw['text'].str.strip().str.len() > 0].copy()
int_empty_removed = int_initial_rows - len(df_clean)
print(f'Removed {int_empty_removed:,} empty posts')

# remove very short posts (less than 20 characters)
int_before_short = len(df_clean)
df_clean = df_clean[df_clean['text'].str.len() >= 20].copy()
int_short_removed = int_before_short - len(df_clean)
print(f'Removed {int_short_removed:,} posts shorter than 20 characters')

# remove duplicate posts
int_before_dupes = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='text').copy()
int_dupes_removed = int_before_dupes - len(df_clean)
print(f'Removed {int_dupes_removed:,} duplicate posts')

# reset index
df_clean = df_clean.reset_index(drop=True)

# add text length column
df_clean['text_length'] = df_clean['text'].str.len()

# add word count column
df_clean['word_count'] = df_clean['text'].str.split().str.len()

int_total_removed = int_initial_rows - len(df_clean)
print(f'\nTotal removed: {int_total_removed:,} ({int_total_removed/int_initial_rows*100:.1f}%)')
print(f'Remaining: {len(df_clean):,} posts')

## Upload to S3

In [ ]:
# write to parquet in memory and upload to S3
buffer = BytesIO()
df_clean.to_parquet(buffer, index=False, engine='pyarrow')
buffer.seek(0)

s3_client.put_object(
    Bucket=str_bucket,
    Key=str_s3_output_path,
    Body=buffer.getvalue(),
    ContentType='application/octet-stream'
)

str_s3_uri = f's3://{str_bucket}/{str_s3_output_path}'
print(f'Successfully uploaded cleaned data to S3')
print(f'  S3 URI: {str_s3_uri}')

## Summary Statistics

In [ ]:
print('\n' + '='*60)
print('DATA COLLECTION SUMMARY')
print('='*60)
print(f'\nDataset: 20 Newsgroups (scikit-learn)')
print(f'Total posts loaded: {int_initial_rows:,}')
print(f'Posts after cleaning: {len(df_clean):,}')
print(f'Posts removed: {int_total_removed:,} ({int_total_removed/int_initial_rows*100:.1f}%)')
print(f'\nCategories: {df_clean["category"].nunique()}')
print(f'Broad groups: {df_clean["broad_category"].nunique()}')
print(f'\nText length (characters):')
print(f'  Mean: {df_clean["text_length"].mean():,.0f}')
print(f'  Median: {df_clean["text_length"].median():,.0f}')
print(f'  Max: {df_clean["text_length"].max():,.0f}')
print(f'\nWord count:')
print(f'  Mean: {df_clean["word_count"].mean():,.0f}')
print(f'  Median: {df_clean["word_count"].median():,.0f}')
print(f'  Max: {df_clean["word_count"].max():,.0f}')
print(f'\nS3 URI: {str_s3_uri}')
print('='*60)